In [3]:
import os
import torch
from torch import nn
from torch.utils.data import Dataset, DataLoader
from torchvision import datasets, transforms
import pandas as pd

In [24]:
def parse_option_size(size_str):
    if isinstance(size_str, str):
        parts = size_str.split(" x ")
        return int(parts[0]), int(parts[1])  # Return as tuple (x, y)
    return 0, 0

class OptionsDataset(Dataset):
    def __init__(self, csv_file):
        self.data = pd.read_csv(csv_file).dropna()

        self.data = self.data.drop(columns=[' [QUOTE_READTIME]', ' [QUOTE_DATE]', ' [EXPIRE_DATE]'], errors='ignore')

        self.data[[' [C_SIZE]', ' [C_SIZE_2]']] = self.data[' [C_SIZE]'].apply(lambda x: pd.Series(parse_option_size(x)))
        self.data[[' [P_SIZE]', ' [P_SIZE_2]']] = self.data[' [P_SIZE]'].apply(lambda x: pd.Series(parse_option_size(x)))
        self.data = self.data.apply(pd.to_numeric, errors='coerce')

        self.X = torch.tensor(self.data.iloc[:, :-1].values, dtype=torch.float64)  # Features
        self.y = torch.tensor(self.data.iloc[:, -1].values, dtype=torch.float64)   # Target 

    def __len__(self):
        return len(self.data)

    def __getitem__(self, idx):
        return self.X[idx], self.y[idx]

In [25]:
path = "C:/Users/kjg23/gitstuff/MachineLearningForEuropeanOptions/optionsdx/aapl_eod_2023q1-fslib7/aapl_eod_202301.csv"

dataset = OptionsDataset(path)
dataloader = DataLoader(dataset, batch_size=32, shuffle=True)

    [C_IV]  [C_VOLUME]  [C_SIZE]  [P_SIZE]  [C_SIZE_2]  [P_SIZE_2]   [P_IV]  \
0  4.35530         0.0        33         0           6         198  2.87776   
1  3.93036         0.0         2         0           2           3  2.59076   
2  3.41421         0.0         2         0          33          16  2.32896   
3  3.24174         0.0        34         0          45          18  2.08767   
4  2.74841         0.0        90         0          36           6  1.86409   

   [P_VOLUME]  
0        33.0  
1         0.0  
2         1.0  
3         0.0  
4         0.0  
<class 'pandas.core.frame.DataFrame'>
RangeIndex: 23472 entries, 0 to 23471
Data columns (total 32 columns):
 #   Column                  Non-Null Count  Dtype  
---  ------                  --------------  -----  
 0   [QUOTE_UNIXTIME]        23472 non-null  int64  
 1    [QUOTE_TIME_HOURS]     23472 non-null  float64
 2    [UNDERLYING_LAST]      23472 non-null  float64
 3    [EXPIRE_UNIX]          23472 non-null  int64  
 4

In [ ]:
for X_batch, y_batch in dataloader:
    print("Feature batch shape:", X_batch.shape)
    print("Target batch shape:", y_batch.shape)
    break